# Volcano ELT Pipeline

Pipeline składa się z trzech kroków:
- **Extract** — pobieramy dane z zewnętrznych źródeł (API, pliki)
- **Load** — zapisujemy dane surowe na dysk (nasz lokalny "data lake")
- **Transform** — czyścimy i przekształcamy dane w pandas

Źródła:
1. **NASA EONET API** — bieżące zdarzenia wulkaniczne (ostatnie 2 lata)
2. **NOAA NGDC API** — historyczne znaczące erupcje wulkanów

---
## Setup

In [2]:
import requests
import json
import pandas as pd
import pathlib

# Tworzymy foldery dla surowych i czystych danych
RAW_DIR = pathlib.Path("raw")
CLEAN_DIR = pathlib.Path("clean")
RAW_DIR.mkdir(exist_ok=True)
CLEAN_DIR.mkdir(exist_ok=True)

print("Foldery gotowe:", list(pathlib.Path(".").iterdir()))

Foldery gotowe: [PosixPath('elt_volcanos.ipynb'), PosixPath('clean'), PosixPath('volcano_elt.ipynb'), PosixPath('raw')]


---
## EXTRACT #1 — NASA EONET: bieżące zdarzenia wulkaniczne

NASA EONET (Earth Observatory Natural Event Tracker) śledzi naturalne zdarzenia na Ziemi w czasie rzeczywistym.
API jest darmowe i nie wymaga klucza.

Parametry zapytania:
- `category=volcanoes` — tylko zdarzenia wulkaniczne
- `status=all` — zarówno aktywne jak i zakończone
- `days=730` — ostatnie 2 lata

In [3]:
EONET_URL = "https://eonet.gsfc.nasa.gov/api/v3/events"

params = {
    "category": "volcanoes",
    "status": "all",
    "limit": 500,
    "days": 730
}

print("Pobieranie danych z NASA EONET...")
response = requests.get(EONET_URL, params=params)

# raise_for_status() rzuca błąd jeśli request się nie powiódł (np. 404, 500)
response.raise_for_status()

eonet_raw = response.json()

print(f"Status: {response.status_code}")
print(f"Pobrano zdarzeń: {len(eonet_raw['events'])}")
print(f"\nPrzykładowe zdarzenie:")
print(json.dumps(eonet_raw['events'][0], indent=2))

Pobieranie danych z NASA EONET...
Status: 200
Pobrano zdarzeń: 72

Przykładowe zdarzenie:
{
  "id": "EONET_20710",
  "title": "Nevados del Chillan Volcano, Chile",
  "description": null,
  "link": "https://eonet.gsfc.nasa.gov/api/v3/events/EONET_20710",
  "closed": null,
  "categories": [
    {
      "id": "volcanoes",
      "title": "Volcanoes"
    }
  ],
  "sources": [
    {
      "id": "SIVolcano",
      "url": "https://volcano.si.edu/volcano.cfm?vn=357070"
    }
  ],
  "geometry": [
    {
      "magnitudeValue": null,
      "magnitudeUnit": null,
      "date": "2026-06-15T00:00:00Z",
      "type": "Point",
      "coordinates": [
        -71.378,
        -36.868
      ]
    }
  ]
}


## LOAD #1 — zapisujemy surowy JSON na dysk

To jest moment "Load" w ELT. Dane trafiają do `raw/` dokładnie takie jak przyszły z API —
bez żadnych modyfikacji. To jest nasz "data lake".

Dlaczego tak? Jeśli później odkryjemy że czegoś nam brakuje albo coś zrobimy źle w transformacji,
zawsze możemy wrócić do surowych danych i zacząć transformację od nowa.

In [ ]:
eonet_path = RAW_DIR / "eonet_volcanoes.json"

with open(eonet_path, "w", encoding="utf-8") as f:
    json.dump(eonet_raw, f, indent=2, ensure_ascii=False)

print(f"Zapisano do: {eonet_path}")
print(f"Rozmiar pliku: {eonet_path.stat().st_size / 1024:.1f} KB")

---
## EXTRACT #2 — NOAA: historyczne erupcje wulkanów

NOAA NGDC (National Geophysical Data Center) ma bazę znaczących zdarzeń geologicznych.
API zwraca dane historyczne o erupcjach — tysiące wpisów, od starożytności do dziś.

In [ ]:
NOAA_URL = "https://www.ngdc.noaa.gov/hazel/hazard-service/api/v1/volcanoEvents"

params_noaa = {
    "minYear": 1900,  # erupcje od 1900 roku
    "orderBy": "year"
}

print("Pobieranie danych z NOAA NGDC...")
response_noaa = requests.get(NOAA_URL, params=params_noaa, timeout=30)
response_noaa.raise_for_status()

noaa_raw = response_noaa.json()

# NOAA zwraca dane w kluczu 'items'
erupcje = noaa_raw.get("items", [])
print(f"Status: {response_noaa.status_code}")
print(f"Pobrano erupcji: {len(erupcje)}")
print(f"\nPrzykładowa erupcja:")
print(json.dumps(erupcje[0], indent=2))

## LOAD #2 — zapisujemy dane NOAA

In [ ]:
noaa_path = RAW_DIR / "noaa_eruptions.json"

with open(noaa_path, "w", encoding="utf-8") as f:
    json.dump(noaa_raw, f, indent=2, ensure_ascii=False)

print(f"Zapisano do: {noaa_path}")
print(f"Rozmiar pliku: {noaa_path.stat().st_size / 1024:.1f} KB")
print(f"\nZawartość folderu raw/:")
for f in RAW_DIR.iterdir():
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

---
## TRANSFORM — wczytujemy surowe dane do pandas i czyścimy

Teraz dopiero zaczynamy analizę. Wcześniej tylko pobieraliśmy i zapisywaliśmy.

Pandas `DataFrame` = tabela danych. Każda kolumna to seria wartości.

In [ ]:
# Wczytujemy surowe dane NOAA z dysku (zawsze z raw/, nie z pamięci)
with open(RAW_DIR / "noaa_eruptions.json", encoding="utf-8") as f:
    noaa_data = json.load(f)

# Tworzymy DataFrame z listy erupcji
df_eruptions = pd.DataFrame(noaa_data["items"])

print(f"Kształt tabeli: {df_eruptions.shape}  (wiersze, kolumny)")
print(f"\nKolumny:")
print(df_eruptions.columns.tolist())
print(f"\nPierwsze 5 wierszy:")
df_eruptions.head()

In [ ]:
# Ile brakujących wartości w każdej kolumnie?
print("Brakujące wartości (NaN):")
print(df_eruptions.isnull().sum().sort_values(ascending=False))

In [ ]:
# Wybieramy tylko kolumny które nas interesują i nadajemy im czytelne nazwy
# (nazwy kolumn sprawdź w output powyżej — mogą się różnić)
kolumny = {
    "year": "rok",
    "month": "miesiac",
    "day": "dzien",
    "name": "nazwa_wulkanu",
    "country": "kraj",
    "latitude": "szerokosc",
    "longitude": "dlugosc",
    "elevationMeters": "wysokosc_m",
    "vei": "vei",           # Volcanic Explosivity Index (0-8, jak skala Richtera)
    "deaths": "ofiary",
    "injuries": "ranni"
}

# Bierzemy tylko te kolumny które istnieją w danych
istniejace = {k: v for k, v in kolumny.items() if k in df_eruptions.columns}
df_clean = df_eruptions[list(istniejace.keys())].rename(columns=istniejace)

print(f"Kolumny po selekcji: {df_clean.columns.tolist()}")
df_clean.head()

In [ ]:
# ANALIZA 1: Które kraje miały najwięcej znaczących erupcji od 1900?
erupcje_kraj = (
    df_clean
    .groupby("kraj")["nazwa_wulkanu"]
    .count()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
    .rename(columns={"nazwa_wulkanu": "liczba_erupcji"})
)

print("Top 15 krajów według liczby erupcji (1900-dziś):")
erupcje_kraj

In [ ]:
# ANALIZA 2: Najsilniejsze erupcje (VEI >= 5)
# VEI 5 = erupcja jak Mount St. Helens 1980
# VEI 6 = Pinatubo 1991
# VEI 7 = Tambora 1815 (rok bez lata)

if "vei" in df_clean.columns:
    potezne = (
        df_clean[df_clean["vei"] >= 5]
        [["rok", "nazwa_wulkanu", "kraj", "vei", "ofiary"]]
        .sort_values("vei", ascending=False)
    )
    print(f"Erupcje VEI >= 5 od 1900: {len(potezne)}")
    potezne
else:
    print("Kolumna VEI niedostępna w tym zbiorze")

In [ ]:
# ANALIZA 3: Aktywność per dekada — czy erupcji przybywa czy ubywa?
df_clean["dekada"] = (df_clean["rok"] // 10 * 10).astype("Int64")

per_dekada = (
    df_clean
    .groupby("dekada")["nazwa_wulkanu"]
    .count()
    .reset_index()
    .rename(columns={"nazwa_wulkanu": "liczba_erupcji"})
)

print("Erupcje per dekada:")
per_dekada

In [ ]:
# Zapisujemy przetransformowane dane do clean/
clean_path = CLEAN_DIR / "eruptions_clean.csv"
df_clean.to_csv(clean_path, index=False, encoding="utf-8")

print(f"Dane wyczyszczone zapisane do: {clean_path}")
print(f"Wierszy: {len(df_clean)}, Kolumn: {len(df_clean.columns)}")
print("\nStruktura folderu projektu:")
for item in sorted(pathlib.Path(".").rglob("*")):
    if ".ipynb_checkpoints" not in str(item) and "venv" not in str(item):
        print(f"  {item}")

---
## Co dalej?

- Wczytać dane EONET (NASA) i połączyć z historycznymi
- Narysować mapę aktywnych wulkanów (matplotlib / folium)
- Zapisać do Parquet zamiast CSV (format kolumnowy, szybszy dla dużych danych)
- Zaplanować automatyczne uruchamianie pipeline'u (Airflow / cron)